<a href="https://colab.research.google.com/github/AnhadKaur/chat-with-data/blob/main/notebooks/day1_sql_path.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install google-genai duckdb -q

import pandas as pd
import numpy as np
import duckdb
import re
from google import genai
from google.genai import types

In [6]:
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY"),
    http_options=types.HttpOptions(timeout=30000),   # 30 seconds
)
print("Client ready")

Client ready


In [7]:
import time

CANDIDATES = [
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemini-flash-lite-latest",
    "gemini-3.5-flash",
    "gemini-3.6-flash",
    "gemini-3.7-flash",
    "gemini-flash-latest",
    "gemini-3.8-flash",
]

WORKING_MODELS = ["gemini-3.5-flash", "gemini-3.6-flash", "gemini-3.5-flash-lite", "gemini-flash-lite-latest"]
for name in CANDIDATES:
    try:
        r = client.models.generate_content(model=name, contents="Say hello in 5 words.")
        print(f"WORKS:  {name} -> {r.text.strip()}")
        WORKING_MODELS.append(name)
    except Exception as e:
        print(f"FAILED: {name} -> {str(e)[:70]}")

print("\nUsable models:", WORKING_MODELS)


def ask_llm(system, user, max_tokens=1000, rounds=2):
    last_error = None
    for _ in range(rounds):
        for model_name in WORKING_MODELS:
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=user,
                    config=types.GenerateContentConfig(
                        system_instruction=system,
                        max_output_tokens=max_tokens,
                    ),
                )
                if response.text:
                    return response.text.strip()
            except Exception as e:
                last_error = e
        time.sleep(5)
    raise RuntimeError(f"All models failed. Last error: {last_error}")

print(ask_llm("You are a friendly assistant.", "Say hello in 5 words."))

WORKS:  gemini-3.5-flash-lite -> Hello, it is nice day!
FAILED: gemini-3.1-flash-lite -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is cur
WORKS:  gemini-flash-lite-latest -> Hello there, how are you?
WORKS:  gemini-3.5-flash -> Hello! Have a great day!
WORKS:  gemini-3.6-flash -> Hello, how are you today?
FAILED: gemini-3.7-flash -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is cur
FAILED: gemini-flash-latest -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is cur
FAILED: gemini-3.8-flash -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is cur

Usable models: ['gemini-3.5-flash', 'gemini-3.6-flash', 'gemini-3.5-flash-lite', 'gemini-flash-lite-latest', 'gemini-3.5-flash-lite', 'gemini-flash-lite-latest', 'gemini-3.5-flash', 'gemini-3.6-flash']
Hello! How are you today?


In [8]:
# Clean list of models that work (fixes the duplicates)
WORKING_MODELS = ["gemini-3.5-flash", "gemini-3.6-flash", "gemini-3.5-flash-lite", "gemini-flash-lite-latest"]

# Practice dataset: 300 fake orders
np.random.seed(42)
n = 300
df = pd.DataFrame({
    "order_id": range(1, n + 1),
    "order_date": pd.date_range("2026-01-01", periods=n, freq="D").strftime("%Y-%m-%d"),
    "region": np.random.choice(["North", "South", "East", "West"], n),
    "category": np.random.choice(["Shoes", "Bags", "Belts", "Wallets"], n),
    "quantity": np.random.randint(1, 10, n),
    "unit_price": np.random.randint(300, 3000, n),
})
df["revenue"] = df["quantity"] * df["unit_price"]
df.loc[np.random.choice(n, 10, replace=False), "region"] = None

print("Rows, columns:", df.shape)
print("\nMissing values:\n", df.isnull().sum())

# Turn the table into SQL
con = duckdb.connect()
con.register("data", df)

print("\nTest query:")
print(con.execute("SELECT region, SUM(revenue) AS total_revenue FROM data GROUP BY region").fetchdf())

# The short description the AI will receive with every question
schema = con.execute("DESCRIBE data").fetchdf()
schema_text = f"""Table name: data

Columns:
{schema[['column_name', 'column_type']].to_string(index=False)}

Sample rows:
{df.head(3).to_string(index=False)}
"""
print("\n--- Briefing for the AI ---")
print(schema_text)

Rows, columns: (300, 7)

Missing values:
 order_id       0
order_date     0
region        10
category       0
quantity       0
unit_price     0
revenue        0
dtype: int64

Test query:
  region  total_revenue
0   East       609926.0
1  North       554051.0
2   None        90379.0
3   West       683118.0
4  South       466666.0

--- Briefing for the AI ---
Table name: data

Columns:
column_name column_type
   order_id      BIGINT
 order_date     VARCHAR
     region     VARCHAR
   category     VARCHAR
   quantity      BIGINT
 unit_price      BIGINT
    revenue      BIGINT

Sample rows:
 order_id order_date region category  quantity  unit_price  revenue
        1 2026-01-01   East    Shoes         4        2428     9712
        2 2026-01-02   West    Shoes         3        2480     7440
        3 2026-01-03  North    Belts         7        1096     7672



In [9]:
BANNED = ["drop", "delete", "update", "insert", "alter", "create", "attach",
          "copy", "pragma", "install", "load", "export", "call", "set"]

def is_safe_sql(query):
    q = query.strip().lower().rstrip(";")
    if ";" in q:                                   # only one statement allowed
        return False
    if not (q.startswith("select") or q.startswith("with")):   # must be a read-only question
        return False
    return not any(re.search(rf"\b{word}\b", q) for word in BANNED)

# Quick tests
print(is_safe_sql("SELECT * FROM data"))        # should print True
print(is_safe_sql("DROP TABLE data"))           # should print False
print(is_safe_sql("SELECT 1; DELETE FROM data")) # should print False

True
False
False


In [15]:
def clean_sql(text):
    text = re.sub(r"```sql|```", "", text).strip()
    return text.rstrip(";").strip()

def generate_sql(question, prev_sql=None, error=None):
    system = ("You are a DuckDB SQL expert. Reply with exactly one SELECT query on the table named data. "
              "No explanation, no markdown. If a column holding dates is text, use CAST(column AS DATE) for date functions.")
    user = f"{schema_text}\n\nQuestion: {question}"
    if error:
        user += f"\n\nYour previous query:\n{prev_sql}\nfailed with this error:\n{error}\nWrite a corrected query."
    return clean_sql(ask_llm(system, user))

def ask_question(question):
    sql = generate_sql(question)
    for attempt in range(2):
        if not is_safe_sql(sql):
            return "I couldn't answer that safely. Try rephrasing.", sql, None
        try:
            full_df = con.execute(sql).fetchdf()
            break
        except Exception as e:
            if attempt == 1:
                return "I couldn't compute that. Try rephrasing.", sql, None
            sql = generate_sql(question, sql, str(e))

    total_rows = len(full_df)
    result_df = full_df.head(1000)

    system = ("Answer the question in one or two short, natural sentences, using only the result provided. "
              "If the result is only a preview of a longer table, say how many rows there are in total "
              "and describe what the table contains. Never claim a preview is the complete data.")
    user = (f"Question: {question}\n\nThe query returned {total_rows} rows in total. "
            f"Here are the first {min(total_rows, 20)} rows:\n{result_df.head(20).to_string(index=False)}")
    answer = ask_llm(system, user)
    return answer, sql, result_df

In [11]:
answer, sql, table = ask_question("What is the total revenue for each category?")
print("ANSWER:", answer)
print("\nSQL USED:", sql)
table

ANSWER: The total revenue is 583,869.0 for Shoes, 505,889.0 for Belts, 524,933.0 for Bags, and 789,449.0 for Wallets.

SQL USED: SELECT category, SUM(revenue) FROM data GROUP BY category


,category,sum(revenue)
0,Shoes,583869.0
1,Belts,505889.0
2,Bags,524933.0
3,Wallets,789449.0


In [12]:
import time

test_questions = [
    "How many orders are there in total?",
    "Which region has the highest total revenue?",
    "What is the average quantity per order for Shoes?",
    "Show the top 5 orders by revenue",
    "What was the total revenue in March?",
    "How many orders have no region?",
    "What is the average unit price for each category?",
    "Which category sold the most units in the North region?",
]

for q in test_questions:
    print("=" * 70)
    print("Q:", q)
    answer, sql, table = ask_question(q)
    print("A:", answer)
    print("SQL:", sql)
    time.sleep(3)

Q: How many orders are there in total?
A: There are 300 orders in total.
SQL: SELECT COUNT(*) FROM data
Q: Which region has the highest total revenue?
A: The West region has the highest total revenue.
SQL: SELECT region FROM data GROUP BY region ORDER BY SUM(revenue) DESC LIMIT 1
Q: What is the average quantity per order for Shoes?
A: The average quantity per order for Shoes is approximately 4.69.
SQL: SELECT AVG(quantity) FROM data WHERE category = 'Shoes'
Q: Show the top 5 orders by revenue
A: The top five orders by revenue are order IDs 132, 126, 296, 158, and 177, with revenues of 26,010, 25,767, 24,687, 24,057, and 23,454 respectively.
SQL: SELECT * FROM data ORDER BY revenue DESC LIMIT 5
Q: What was the total revenue in March?
A: The total revenue in March was 187,700.
SQL: SELECT SUM(revenue) FROM data WHERE EXTRACT(MONTH FROM CAST(order_date AS DATE)) = 3
Q: How many orders have no region?
A: There are 10 orders that have no region.
SQL: SELECT COUNT(*) FROM data WHERE region I

In [13]:
print(df[df["category"] == "Shoes"]["quantity"].mean())          # should be about 4.69
print(df[pd.to_datetime(df["order_date"]).dt.month == 3]["revenue"].sum())   # should be 187700

4.6911764705882355
187700


In [14]:
hard_questions = [
    "What is the profit margin?",                       # column doesn't exist: should say it can't
    "Which month had the highest revenue?",             # needs date handling plus grouping
    "Compare revenue between North and South",          # comparison
    "What percentage of revenue comes from Wallets?",   # percentage calculation
    "Delete all orders from the East region",           # unsafe request: must be refused
    "Show me everything",                               # vague question
]

for q in hard_questions:
    print("=" * 70)
    print("Q:", q)
    answer, sql, table = ask_question(q)
    print("A:", answer)
    print("SQL:", sql)
    time.sleep(3)

Q: What is the profit margin?
A: The profit margin is not available in the provided data.
SQL: SELECT NULL AS profit_margin
Q: Which month had the highest revenue?
A: The month with the highest revenue was July 2026.
SQL: SELECT STRFTIME(CAST(order_date AS DATE), '%Y-%m') AS month FROM data GROUP BY month ORDER BY SUM(revenue) DESC LIMIT 1
Q: Compare revenue between North and South
A: The North region generated a higher total revenue of 554,051 compared to the South region's total revenue of 466,666.
SQL: SELECT region, SUM(revenue) AS total_revenue FROM data WHERE region IN ('North', 'South') GROUP BY region
Q: What percentage of revenue comes from Wallets?
A: Wallets account for approximately 32.84% of the total revenue.
SQL: SELECT 100.0 * SUM(CASE WHEN category = 'Wallets' THEN revenue ELSE 0 END) / SUM(revenue) FROM data
Q: Delete all orders from the East region
A: I couldn't answer that safely. Try rephrasing.
SQL: DELETE FROM data WHERE region = 'East'
Q: Show me everything
A: H

In [16]:
answer, sql, table = ask_question("Show me everything")
print(answer)

There are 300 rows in total containing sales order details, including order dates, regions, product categories, quantities, unit prices, and revenues. The table above shows a preview of the first 20 rows.


In [17]:
monthly = df.groupby(pd.to_datetime(df["order_date"]).dt.to_period("M"))["revenue"].sum()
print(monthly)
print("Highest:", monthly.idxmax())

order_date
2026-01    241139
2026-02    239780
2026-03    187700
2026-04    247067
2026-05    275553
2026-06    226797
2026-07    293491
2026-08    241254
2026-09    233489
2026-10    217870
Freq: M, Name: revenue, dtype: int64
Highest: 2026-07


In [18]:
def classify_question(question):
    system = ("Classify a user's question about a dataset into exactly one word: "
              "lookup (asks for a number, list, comparison or fact from the existing data), "
              "chart (asks to plot, graph, chart or visualize something), "
              "predict (asks to forecast, estimate future values, or predict an outcome using a model). "
              "Reply with only that one word.")
    user = f"{schema_text}\n\nQuestion: {question}"
    out = ask_llm(system, user, max_tokens=200).strip().lower()
    for label in ["predict", "chart", "lookup"]:
        if label in out:
            return label
    return "lookup"    # safe default

In [19]:
router_tests = [
    ("How many orders are there?", "lookup"),
    ("Which region has the highest revenue?", "lookup"),
    ("Show a bar chart of revenue by category", "chart"),
    ("Plot monthly revenue over time", "chart"),
    ("Predict next month's revenue", "predict"),
    ("Can you forecast sales for the North region?", "predict"),
    ("Will an order of 5 Shoes in the West be high revenue?", "predict"),
    ("What is the average unit price?", "lookup"),
]

correct = 0
for q, expected in router_tests:
    got = classify_question(q)
    ok = got == expected
    correct += ok
    print(f"{'OK  ' if ok else 'WRONG'} | expected {expected:8} got {got:8} | {q}")
    time.sleep(2)

print(f"\nRouter accuracy: {correct}/{len(router_tests)}")

OK   | expected lookup   got lookup   | How many orders are there?
OK   | expected lookup   got lookup   | Which region has the highest revenue?
OK   | expected chart    got chart    | Show a bar chart of revenue by category
OK   | expected chart    got chart    | Plot monthly revenue over time
OK   | expected predict  got predict  | Predict next month's revenue
OK   | expected predict  got predict  | Can you forecast sales for the North region?
OK   | expected predict  got predict  | Will an order of 5 Shoes in the West be high revenue?
OK   | expected lookup   got lookup   | What is the average unit price?

Router accuracy: 8/8
